# The dark families at 90% identity

`dark_families.ipynb` clustered the dark plasmid proteome at **30% identity / 80% coverage** —
a homology criterion. This notebook uses the strictest clustering in the sweep,
**90% identity / 80% coverage**, which is UniRef90's rule. At that threshold a "family" is
close to *the same protein*, so it answers a different question: not what is homologous, but
what is genuinely, repeatedly recurring.

Produced by `scripts/recluster_sweep.sbatch`:

```
mmseqs easy-cluster --min-seq-id 0.9 -c 0.8 --cov-mode 0 --cluster-reassign 1
```

One clarification on the axis: **coverage is 80% in every run of the sweep, and only identity
varies.** 80% of both sequences is already stricter than anything in the comparable literature
— the *Nature* 2024 novel-family catalogue used 50% of the shorter one — so there was nothing
to gain by tightening it further (`reports/clustering_threshold_sensitivity.md` §2).

In [ ]:
# The dark plasmid families at 90% identity — the most conservative clustering in the sweep.
#
# Produced by scripts/recluster_sweep.sbatch:
#     mmseqs easy-cluster --min-seq-id 0.9 -c 0.8 --cov-mode 0 --cluster-reassign 1
# 90% identity over 80% of BOTH sequences is UniRef90's rule. Note the axis that varies across
# the sweep is IDENTITY; coverage is held at 80% throughout, because that is already stricter
# than anything in the comparable literature (reports/clustering_threshold_sensitivity.md §2).
#
# At this threshold a "family" is close to "the same protein", so it answers a different
# question from dark_families.ipynb: not "what is homologous" but "what is actually recurring".
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RC, PP = ROOT / "data" / "dark_orf_run" / "recluster", ROOT / "data" / "plasmidscope_primary"

S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"


def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)


F90 = pd.read_csv(RC / "dark90_fampfam.tsv", sep="\t")
CL90 = pd.read_csv(RC / "dark90_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
CL90["plasmid_id"] = CL90["member"].str.rsplit("|", n=1).str[0]
MET = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False,
                  usecols=["plasmid_id", "hab_sub", "mob_cluster", "size_bp",
                           "mob_rep_types"]).set_index("plasmid_id")
ORG = pd.read_csv(PP / "working_set.tsv", sep="\t", low_memory=False,
                  usecols=["plasmid_id", "organism"]).set_index("plasmid_id")["organism"]

NORF = 378_552
assert F90["proteins"].sum() == NORF == len(CL90)
CARRIERS = CL90["plasmid_id"].nunique()
print(f"90% clustering: {len(F90):,} families over {NORF:,} dark ORFs on {CARRIERS:,} plasmids")
print(f"  singletons: {(F90['proteins'] == 1).sum():,} families "
      f"({(F90['proteins'] == 1).mean() * 100:.1f}%)")
print(f"  Pfam-named: {F90['pfam_hit'].sum():,} families ({F90['pfam_hit'].mean() * 100:.1f}%), "
      f"covering {F90.loc[F90['pfam_hit'], 'proteins'].sum() / NORF * 100:.1f}% of dark ORFs")

## 1. What tightening the definition does

Against the 30% clustering the other notebooks use, so the cost of the conservative choice is
explicit rather than implied.

In [ ]:
# Q1 — what does tightening 30% -> 90% actually do? Compared against the 30% clustering the
# other notebooks use, so the cost of the conservative choice is explicit.
F30 = pd.read_csv(ROOT / "data" / "dark_orf_run" / "darkfam_stats.tsv", sep="\t").merge(
    pd.read_csv(ROOT / "data" / "pfam_run" / "darkfam_pfam.tsv",
                sep="\t")[["rep", "pfam_hit"]], on="rep", how="left")
F30["pfam_hit"] = F30["pfam_hit"].fillna(False)

CMP = pd.DataFrame({
    "30% (dark_families.ipynb)": {
        "families": len(F30),
        "families on 1 plasmid %": (F30["plasmids"] == 1).mean() * 100,
        "dark ORFs recurrent %": F30.loc[F30["plasmids"] >= 2, "proteins"].sum() / NORF * 100,
        "dark ORFs Pfam-named %": F30.loc[F30["pfam_hit"], "proteins"].sum() / NORF * 100,
        "largest family (proteins)": F30["proteins"].max(),
        "most plasmids": F30["plasmids"].max(),
        "core families": len(F30[(~F30["pfam_hit"]) & (F30["plasmids"] >= 100)
                                 & (F30["lineages"] >= 10)]),
    },
    "90% (this notebook)": {
        "families": len(F90),
        "families on 1 plasmid %": (F90["plasmids"] == 1).mean() * 100,
        "dark ORFs recurrent %": F90.loc[F90["plasmids"] >= 2, "proteins"].sum() / NORF * 100,
        "dark ORFs Pfam-named %": F90.loc[F90["pfam_hit"], "proteins"].sum() / NORF * 100,
        "largest family (proteins)": F90["proteins"].max(),
        "most plasmids": F90["plasmids"].max(),
        "core families": len(F90[(~F90["pfam_hit"]) & (F90["plasmids"] >= 100)
                                 & (F90["lineages"] >= 10)]),
    }})
print(CMP.round(1).to_string())
print("\nThe recurrence claim survives: even when a family means near-identical proteins,")
print(f"{F90.loc[F90['plasmids'] >= 2, 'proteins'].sum() / NORF * 100:.0f}% of dark ORFs are still "
      f"on two or more plasmids. What collapses is the target set.")

BANDS = [(1, 1), (2, 4), (5, 9), (10, 49), (50, 99), (100, 499), (500, 10 ** 9)]
LABS = ["1", "2-4", "5-9", "10-49", "50-99", "100-499", "500+"]
REC = pd.DataFrame([{"band": lab, "families": len(F90[F90["plasmids"].between(lo, hi)]),
                     "dark ORFs": F90.loc[F90["plasmids"].between(lo, hi), "proteins"].sum()}
                    for (lo, hi), lab in zip(BANDS, LABS)]).set_index("band")
REC["% dark ORFs"] = REC["dark ORFs"] / NORF * 100
print("\nfamilies by distinct plasmids, at 90%:")
print(REC.round(1).to_string())

In [ ]:
# Figure 1. Grey = the 30% clustering the other notebooks use; slot 1 = this 90% clustering.
# One meaning for grey throughout the figure.
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), gridspec_kw={"wspace": 0.3})
KEYS = [("dark ORFs recurrent %", "Survives the tightening",
         "dark ORFs in families on >= 2 plasmids", "%"),
        ("dark ORFs Pfam-named %", "Unmoved", "dark ORFs in Pfam-named families", "%"),
        ("core families", "Collapses", "Pfam-negative, >=100 plasmids, >=10 lineages", "n")]
for ax, (k, ttl, sub, unit) in zip(axes, KEYS):
    v = CMP.loc[k].values
    ax.bar([0, 1], v, width=0.5, color=[NEUT, S1])
    for i, val in enumerate(v):
        ax.text(i, val + v.max() * 0.03, f"{val:,.0f}{'%' if unit == '%' else ''}",
                ha="center", fontsize=10, color=INK2)
    ax.set(xticks=[0, 1], xticklabels=["30%", "90%"], yticks=[], ylim=(0, v.max() * 1.22))
    ax.set_title(f"{ttl}\n{sub}", fontsize=10, color=INK, loc="left")
    tidy(ax)
fig.suptitle("Tightening the family definition from 30% to 90% identity",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.05)
plt.show()

## 2. The most prevalent families at 90%

"Prevalent" means the number of distinct plasmids carrying the family.

In [ ]:
# Q2 — the most prevalent families at 90%. "Prevalent" = distinct plasmids carrying the family.
N = 20
TOP = F90.sort_values("plasmids", ascending=False).head(N).copy()
rows = []
for _, r in TOP.iterrows():
    pl = CL90.loc[CL90["rep"].eq(r["rep"]), "plasmid_id"].unique()
    md, og = MET.reindex(pl), ORG.reindex(pl)
    rows.append({"plasmids": int(r["plasmids"]), "lineages": int(r["lineages"]),
                 "habitats": int(r["habitats"]), "len_aa": int(r["rep_len_aa"]),
                 "pfam": r["pfam_name"] if r["pfam_hit"] else "—",
                 "med_bp": int(md["size_bp"].median()),
                 "top_hab": md["hab_sub"].value_counts().index[0][:22]
                 if md["hab_sub"].notna().any() else "-",
                 "host": og.value_counts().index[0] if og.notna().any() else "(none named)"})
TT = pd.DataFrame(rows, index=TOP["rep"].values)
print(f"the {N} most prevalent dark families at 90% identity:")
print(TT.to_string())
print(f"\nPfam-negative: {(TT['pfam'] == '—').sum()} of {N}")
print(f"median lineages spanned: {TT['lineages'].median():.0f}  "
      f"(at 30% the single most prevalent family spanned 88)")

In [ ]:
# Figure 2. Grey = the family has a Pfam name, i.e. it is explained; slot 1 = still unnamed.
t = TT.iloc[::-1]
y = np.arange(len(t))
cols = [NEUT if p != "—" else S1 for p in t["pfam"]]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 6.4),
                               gridspec_kw={"wspace": 0.05, "width_ratios": [1.5, 1]})
axA.barh(y, t["plasmids"], height=0.6, color=cols)
for i, (v, p, ln, bp) in enumerate(zip(t["plasmids"], t["pfam"], t["len_aa"], t["med_bp"])):
    axA.text(v + TT["plasmids"].max() * 0.012, i, f"{v:,}   {ln} aa   {p}   ·  carriers ~{bp:,} bp",
             va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=[r.replace("IMGPR_plasmid_", "IMGPR_")[:30] for r in t.index],
        xticks=[], xlim=(0, TT["plasmids"].max() * 1.5))
axA.tick_params(axis="y", labelsize=7)
axA.set_xlabel("distinct plasmids carrying the family", fontsize=9, color=INK2)
axA.set_title(f"A — most prevalent at 90% identity\ngrey: {(TT['pfam'] != '—').sum()} of {N} have "
              f"a Pfam name", fontsize=10.5, color=INK, loc="left")
tidy(axA)

axB.barh(y, t["lineages"], height=0.6, color=cols)
for i, v in enumerate(t["lineages"]):
    axB.text(v + 0.12, i, f"{v}", va="center", fontsize=8.5, color=INK2)
axB.set(yticks=y, yticklabels=[], xlim=(0, TT["lineages"].max() * 1.35), ylim=axA.get_ylim())
axB.set_xlabel("MOB primary clusters spanned", fontsize=9, color=INK2)
axB.set_title("B — and how many lineages they reach\nnote the scale: single digits",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("The most prevalent dark families under the most conservative clustering",
             fontsize=12, color=INK, x=0.006, ha="left", y=0.98)
plt.show()

## 3. Those carriers are one plasmid, sequenced many times

Every one of the top five reports carriers of ~4,247 bp and 2–3 MOB lineages. That is a
testable pattern, not a coincidence.

In [ ]:
# Q3 — those top families all report carriers of ~4,247 bp and 2-3 lineages. Test the obvious
# explanation directly: are they riding one replicon? Same test as widespread_dark_orfs.ipynb
# section 3, now at family rather than exact-sequence level.
K = 6
sets = [set(CL90.loc[CL90["rep"].eq(r), "plasmid_id"]) for r in TOP["rep"].head(K)]
OV = pd.DataFrame([[len(a & b) / min(len(a), len(b)) * 100 for b in sets] for a in sets],
                  index=[r[-18:] for r in TOP["rep"].head(K)],
                  columns=[r[-18:] for r in TOP["rep"].head(K)])
print(f"carrier-set overlap of the top {K} families (% of the smaller set):")
print(OV.round(0).to_string())

# The matrix has visible block structure: families 1-5 share carriers, family 6 shares none of
# them. That is the control -- it shows the overlap is a property of specific families, not an
# artifact of the measure. The intersection is therefore taken over the block, not all K.
BLOCK = 5
core = set.intersection(*sets[:BLOCK])
cm = MET.reindex(sorted(core))
print(f"\nplasmids carrying all {BLOCK} of the overlapping block: {len(core):,}")
print(f"  (family {K} shares {len(sets[K - 1] & sets[0])} carriers with family 1 — a different replicon, median {MET.reindex(sorted(sets[K - 1]))['size_bp'].median():,.0f} bp)")
if len(core):
    print(f"  size          median {cm['size_bp'].median():,.0f} bp  "
          f"IQR {cm['size_bp'].quantile(.25):,.0f}-{cm['size_bp'].quantile(.75):,.0f}")
    print(f"  MOB clusters  {cm['mob_cluster'].nunique()}   "
          f"replicon no-call {cm['mob_rep_types'].isna().mean() * 100:.0f}%")
    print(f"  dominant habitat {cm['hab_sub'].value_counts().index[0]} "
          f"({cm['hab_sub'].value_counts(normalize=True).iloc[0] * 100:.0f}%)")
print("\nSo the prevalence ranking at 90% is dominated by one heavily-sequenced small plasmid.\n"
      "Tightening the threshold does not remove the sampling confound — it sharpens it, because\n"
      "near-identical clustering tracks clonal expansion almost exactly. Lineage spread, not\n"
      "plasmid count, is what separates biology from sequencing effort.")

In [ ]:
# Figure 3. Sequential ramp for a magnitude (overlap); no categorical colour, so grey is unused
# in this figure.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 4.8),
                               gridspec_kw={"wspace": 0.3, "width_ratios": [1, 1.1]})
im = axA.imshow(OV.values, cmap="Blues", vmin=0, vmax=100)
for i in range(K):
    for j in range(K):
        v = OV.values[i, j]
        axA.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8.5,
                 color="white" if v > 55 else INK)
axA.set(xticks=range(K), yticks=range(K), xticklabels=OV.columns, yticklabels=OV.index)
axA.tick_params(labelsize=7, colors=INK2, length=0)
plt.setp(axA.get_xticklabels(), rotation=40, ha="right")
for s in axA.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=axA, fraction=0.046, pad=0.04).set_label("% of smaller carrier set",
                                                             fontsize=8.5, color=INK2)
axA.set_title(f"A — the top {K} share their carriers\ncarrier-set overlap",
              fontsize=10.5, color=INK, loc="left")

axB.scatter(F90.loc[F90["plasmids"] >= 20, "plasmids"],
            F90.loc[F90["plasmids"] >= 20, "lineages"], s=14, color=S1, alpha=0.65, linewidth=0)
axB.set(xscale="log", yscale="log")
axB.set_xlabel("distinct plasmids (log)", fontsize=9, color=INK2)
axB.set_ylabel("MOB primary clusters spanned (log)", fontsize=9, color=INK2)
rho = F90.loc[F90["plasmids"] >= 20, ["plasmids", "lineages"]].corr(method="spearman").iloc[0, 1]
axB.set_title(f"B — prevalence does not imply spread\nSpearman rho = {rho:.2f} over the "
              f"{(F90['plasmids'] >= 20).sum():,} families on >= 20 plasmids",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("Why the prevalence ranking is still a sequencing artifact at 90% identity",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.02)
plt.show()

## 4. The families that survive every filter

The target-set definition from `dark_families.ipynb` §5 — Pfam-negative, on ≥100 plasmids,
spanning ≥10 MOB lineages — applied at the strictest clustering. At 30% it returned 93
families.

In [ ]:
# Q4 — apply the target-set filter from dark_families.ipynb at this threshold: Pfam-negative,
# on >= 100 plasmids, spanning >= 10 MOB lineages. At 30% that returned 93 families. Here it is
# the strictest form of the same question.
MIN_PL, MIN_LIN = 100, 10
CORE = (F90[(~F90["pfam_hit"]) & (F90["plasmids"] >= MIN_PL) & (F90["lineages"] >= MIN_LIN)]
        .sort_values("lineages", ascending=False).reset_index(drop=True))
print(f"CORE at 90% identity = Pfam-negative, >= {MIN_PL} plasmids, >= {MIN_LIN} lineages")
print(f"  {len(CORE)} families, {CORE['proteins'].sum():,} dark ORFs "
      f"({CORE['proteins'].sum() / NORF * 100:.2f}% of the dark proteome)\n")

seqs, name, cur = {}, None, []
for line in open(RC / "dark90_rep_seq.fasta"):
    if line[0] == ">":
        if name is not None:
            seqs[name] = "".join(cur)
        name, cur = line[1:].strip().split()[0], []
    else:
        cur.append(line.strip())
seqs[name] = "".join(cur)

det = []
for _, r in CORE.iterrows():
    pl = CL90.loc[CL90["rep"].eq(r["rep"]), "plasmid_id"].unique()
    md, og = MET.reindex(pl), ORG.reindex(pl)
    det.append({"rep": r["rep"], "proteins": int(r["proteins"]), "plasmids": int(r["plasmids"]),
                "lineages": int(r["lineages"]), "habitats": int(r["habitats"]),
                "len_aa": int(r["rep_len_aa"]), "med_bp": int(md["size_bp"].median()),
                "top_hab": md["hab_sub"].value_counts().index[0]
                if md["hab_sub"].notna().any() else "-",
                "host": og.value_counts().index[0] if og.notna().any() else "(none named)",
                "seq": seqs[r["rep"]]})
DET = pd.DataFrame(det)
print(DET.drop(columns="seq").to_string(index=False))

OUT = ROOT / "data" / "dark_orf_run" / "recluster"
DET.drop(columns="seq").to_csv(OUT / "core_dark_families_90.tsv", sep="\t", index=False)
with open(OUT / "core_dark_families_90.faa", "w") as fh:
    for _, r in DET.iterrows():
        fh.write(f">{r['rep']} plasmids={r['plasmids']} lineages={r['lineages']} "
                 f"habitats={r['habitats']} len={r['len_aa']}\n{r['seq']}\n")
print(f"\nwritten -> {(OUT / 'core_dark_families_90.tsv').relative_to(ROOT)} and .faa")

print("\nThe five sequences:")
for _, r in DET.iterrows():
    print(f"\n>{r['rep']}   {r['len_aa']} aa | {r['plasmids']} plasmids | "
          f"{r['lineages']} lineages | {r['habitats']} habitats | host {r['host']}")
    for i in range(0, len(r["seq"]), 60):
        print("  " + r["seq"][i:i + 60])

In [ ]:
# Figure 4. Slot 1 throughout — one series, the surviving families, so no colour semantics are
# in play and grey is unused.
d = DET.iloc[::-1]
y = np.arange(len(d))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4), gridspec_kw={"wspace": 0.55})
for ax, col, lab in ((axes[0], "lineages", "MOB primary clusters spanned"),
                     (axes[1], "habitats", "habitats occupied"),
                     (axes[2], "plasmids", "distinct plasmids")):
    ax.barh(y, d[col], height=0.5, color=S1)
    for i, v in enumerate(d[col]):
        ax.text(v + d[col].max() * 0.03, i, f"{v:,}", va="center", fontsize=9, color=INK2)
    ax.set(yticks=y, xticks=[], xlim=(0, d[col].max() * 1.25))
    ax.set_yticklabels([f'{r.replace("IMGPR_plasmid_", "IMGPR_")}\n{n} aa'
                        for r, n in zip(d["rep"], d["len_aa"])], fontsize=6.5)
    ax.set_xlabel(lab, fontsize=9, color=INK2)
    tidy(ax)
fig.suptitle(f"The {len(DET)} families that survive every filter\n"
             f"90% identity / 80% coverage · no Pfam-A hit at --cut_ga · >= 100 plasmids · "
             f">= 10 MOB lineages", fontsize=12, color=INK, x=0.006, ha="left", y=1.16)
plt.show()

## What the 90% view shows

**The recurrence result holds.** Even when a family means *near-identical protein*, **70.3% of
dark ORFs sit in families found on two or more plasmids**, and 137 families are on 100+
plasmids each. The central claim of `dark_plasmidome.ipynb` does not depend on permissive
clustering.

**The Pfam verdict is unmoved.** 38.9% of dark ORFs are in Pfam-named families here, against
39.4% at 30%. The artifact-versus-real conclusion is threshold-independent.

**The prevalence ranking is still a sequencing artifact — more so, not less.** The five most
prevalent families overlap on 70–97% of their carriers, and **477 plasmids carry all five**, at
~4,247 bp with 2–3 MOB lineages. Tightening the threshold sharpens the confound rather than
removing it, because near-identical clustering tracks clonal expansion almost exactly. The
sixth-ranked family shares *no* carriers with those five and sits on a ~2,857 bp replicon —
which is the control showing the overlap is real and family-specific.

**And the target set collapses to five families.** 93 at 30% identity, 61 once the clustering
criterion is properly enforced, 39 at UniRef50, **5 here** — 788 dark ORFs, 0.21% of the dark
proteome.

### The five survivors

| length | plasmids | MOB lineages | habitats | commonest host |
|---:|---:|---:|---:|---|
| 70 aa | 138 | **42** | 12 | *Lactiplantibacillus plantarum* |
| 49 aa | 227 | 30 | **20** | *Escherichia coli* |
| 121 aa | 101 | 30 | 11 | *Lactiplantibacillus plantarum* |
| 105 aa | 121 | 15 | 16 | *Acinetobacter pittii* |
| 78 aa | 186 | 14 | **20** | *Klebsiella pneumoniae* |

These are short proteins held at **≥90% amino-acid identity across 14–42 independent MOB
lineages and up to 20 habitats**, with no Pfam-A match at the curated gathering threshold. That
combination is hard to explain by sampling: clonal expansion produces high identity within one
lineage, not across forty. Sequence conservation that strong across that many lineages is the
signature of purifying selection, which means these proteins are doing something.

Sequences and metadata: `data/dark_orf_run/recluster/core_dark_families_90.faa` and `.tsv`.

### How to read this against the other notebooks

The 30% and 90% views are not competing answers, they are different questions. 30% asks what is
*homologous* and gives a picture of the dark proteome's breadth; 90% asks what is *conserved*
and gives a much smaller set with a much stronger claim attached to each member. If the point
is a shortlist for structure prediction and experimental work, these five — not the 93 — are
where the evidence is strongest per candidate.